# 07 — Create Subscription Orders (active)

The final creation step for ACTIVE subscriptions — the ones that went through
the full Voyager address lookup. For inactive subscriptions, see
`07_Create_Inactive_Subscription_Orders.ipynb`.

`build_subscription_order_payload`, `create_order_for_subscription`, and
`create_all_orders` now live in `onebill_common.py`, shared with the inactive
notebook — this notebook just loads the active-pipeline data and calls them.

For every subscription that got an address in `06_Create_Addresses.ipynb`:

1. Resolve its plan — `03_Match_Plan_Codes.ipynb` mapping, falling back to
   `STATIC_FALLBACK_PLAN` if unmatched.
2. Optionally attach a primary-contact summary from
   `01_Fetch_Contacts.ipynb` (see `ATTACH_CONTACT_SUMMARY_TO_ORDER`).
3. Build the order payload per the field mapping below.
4. `POST /rest/OrderService/v1/order`.

**vBill -> OneBill field mapping**

| OneBill field | Source |
|---|---|
| Subscription Username | vBill Subscription Label |
| External Service ID | Supplier Service ID |
| Imported Subscription USN | Subscription USN |
| Subscription USN | *(autogenerated by OneBill — not sent)* |
| Radius Username | Voyager `radiusUsers[0]` (from `05_Fetch_Subscriptions.ipynb`'s circuits lookup), falling back to vBill Subscription Label if Voyager didn't return one |
| Activation date | Subscription Start Date |
| Recurring-from date | fixed `2026-08-01` |
| Term | `0` if `SubscriptionEndDate` and `NextPlanStartDate` are both null, else `1` |
| followOnTermDetails.term | (only when Term=1) whole months between today and `SubscriptionEndDate` (falling back to `NextPlanStartDate` if the end date is null) |
| Vendor | `Supplier` mapped via `SUPPLIER_TO_VENDOR` (Chorus/Enable/UFF -> chorus/enable/tff); omitted + logged if `Supplier` is set but unmapped |

> **NOTE**: `resolve_vendor()` reads `subscription["Supplier"]` — this notebook pulls `df_subscriptions` via `SELECT *` from `bi_datastore.billing_subscription`, so double-check that's the exact column name in your MySQL table (vs. e.g. `SupplierName`) before a full run; if it's different, update the `.get("Supplier")` calls in `build_subscription_order_payload`.

## 1. Setup — load everything the previous notebooks produced

In [9]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403
from concurrent.futures import ThreadPoolExecutor, as_completed

logger = get_logger("create_subscription_orders")

df_subscriptions   = load_subscriptions_resolved()
df_address_results = load_df("address_results", dtype={"ship_add_id": str})
df_plan_mapping     = try_load_df("plan_mapping", dtype=str)
df_contacts          = try_load_df("contacts")

if df_plan_mapping is None:
    logger.warning("03_plan_code_mapping.csv not found — every subscription falls back to STATIC_FALLBACK_PLAN.")
    df_plan_mapping = pd.DataFrame(columns=["PlanCode", "product_name", "priceplan_name"])

if df_contacts is None and ATTACH_CONTACT_SUMMARY_TO_ORDER:
    logger.warning("01_contacts_by_account.csv not found — orders will be created without a contact summary.")

logger.info(
    f"{len(df_subscriptions):,} subscriptions, {len(df_address_results):,} address results, "
    f"{len(df_plan_mapping):,} plan mappings, "
    f"{len(df_contacts) if df_contacts is not None else 0:,} contact summaries"
)


2026-07-27 11:06:00,565 [INFO] 264 subscriptions, 264 address results, 9 plan mappings, 37,552 contact summaries


## 2. Join subscriptions with their address result

In [10]:
df_work = df_subscriptions.merge(
    df_address_results[["SubscriptionUSN", "status", "ship_add_id", "error"]].rename(
        columns={"status": "AddressStatus", "error": "AddressError"}
    ),
    on="SubscriptionUSN", how="left",
)

ready = df_work[df_work["AddressStatus"].isin(["created", "exists"])]
not_ready = df_work[~df_work["AddressStatus"].isin(["created", "exists"])]
logger.info(f"{len(ready):,} subscriptions have an address and are ready for order creation; "
            f"{len(not_ready):,} do not and will be skipped")
not_ready[["SubscriptionUSN", "AddressStatus", "AddressError"]].head(20)


2026-07-27 11:06:00,711 [INFO] 247 subscriptions have an address and are ready for order creation; 17 do not and will be skipped


,SubscriptionUSN,AddressStatus,AddressError
1,V113062335,skipped,no SupplierServiceID on this subscription
2,V113062343,skipped,no SupplierServiceID on this subscription
5,V113062400,skipped,circuits lookup failed: HTTPSConnectionPool(ho...
6,V113062418,skipped,circuits lookup failed: HTTPSConnectionPool(ho...
7,V113062632,skipped,circuits lookup failed: HTTPSConnectionPool(ho...
8,V113062616,skipped,circuits lookup failed: HTTPSConnectionPool(ho...
9,V113062962,skipped,circuits lookup failed: HTTPSConnectionPool(ho...
10,V113062970,skipped,circuits lookup failed: HTTPSConnectionPool(ho...
11,V113062988,skipped,circuits lookup failed: HTTPSConnectionPool(ho...
43,V113064133,skipped,circuits lookup failed: ('Connection aborted.'...


## 3. Plan resolution

In [11]:
resolve_plan = make_plan_resolver(df_plan_mapping)


## 4. Contact-summary lookup (optional)

Joins each subscription's *original* vBill `AccountCode` (not the bucket
account) against `01_Fetch_Contacts.ipynb`'s output.

In [12]:
contact_summary_attributes = make_contact_summary_fn(df_contacts)


## 5. Build the order payload (field mapping)

In [13]:
# build_subscription_order_payload is now defined in onebill_common.py (shared with
# 07_Create_Inactive_Subscription_Orders.ipynb) — nothing to define here.


## 6. Per-subscription worker

In [14]:
# create_order_for_subscription is now defined in onebill_common.py (shared with
# 07_Create_Inactive_Subscription_Orders.ipynb) — nothing to define here.


## 7. Run (parallel driver)

In [15]:
order_results_df = create_all_orders(ready, resolve_plan, contact_summary_attributes)
order_results_df.head(20)


2026-07-27 11:06:04,062 [INFO] Creating orders for 247 subscriptions with 10 workers...


2026-07-27 11:06:25,960 [INFO] [OK] subscription V113062392 -> 99965692_fullbatch5 (plan_matched=True, orderId=unknown)
2026-07-27 11:06:27,168 [INFO] [OK] subscription V113063044 -> 99965692_fullbatch5 (plan_matched=True, orderId=unknown)
2026-07-27 11:06:28,346 [INFO] [OK] subscription V113063085 -> 99965692_fullbatch5 (plan_matched=True, orderId=unknown)
2026-07-27 11:06:29,784 [INFO] [OK] subscription V113063036 -> 99965692_fullbatch5 (plan_matched=True, orderId=unknown)
2026-07-27 11:06:31,043 [INFO] [OK] subscription V113062996 -> 99965692_fullbatch5 (plan_matched=True, orderId=unknown)
2026-07-27 11:06:32,183 [INFO] [OK] subscription V113061451 -> 99965692_fullbatch5 (plan_matched=True, orderId=unknown)
2026-07-27 11:06:33,366 [INFO] [OK] subscription V113062384 -> 99965692_fullbatch5 (plan_matched=True, orderId=unknown)
2026-07-27 11:06:34,609 [INFO] [OK] subscription V113063002 -> 99965692_fullbatch5 (plan_matched=True, orderId=unknown)
2026-07-27 11:06:35,953 [INFO] [OK] subs

,SubscriptionUSN,TargetAccountNumber,status,plan_matched,productName,priceplanName,onebill_order_id,error
0,V113062392,99965692_fullbatch5,success,True,Wholesale Fibre BS2 (TFF),WS Tail+Data - BS2 Res Fibre Starter (TFF) - 1...,unknown,None
1,V113063044,99965692_fullbatch5,success,True,Wholesale Fibre BS2 (Chorus),WS Tail+Data - BS2 Res Fibre Starter (Chorus) ...,unknown,None
2,V113063085,99965692_fullbatch5,success,True,Wholesale Fibre BS2 (Chorus),WS Tail+Data - BS2 Res Fibre Starter (Chorus) ...,unknown,None
3,V113063036,99965692_fullbatch5,success,True,Wholesale Fibre BS2 (Chorus),WS Tail+Data - BS2 Res Fibre Starter (Chorus) ...,unknown,None
4,V113062996,99965692_fullbatch5,success,True,Wholesale Fibre BS2 (Chorus),WS Tail+Data - BS2 Res Fibre Starter (Chorus) ...,unknown,None
5,V113061451,99965692_fullbatch5,success,True,Wholesale Fibre BS2 (Enable),WS Tail+Data - BS2 Res (Enable) - 920/500,unknown,None
6,V113062384,99965692_fullbatch5,success,True,Wholesale Fibre BS2 (TFF),WS Tail+Data - BS2 Res Fibre Starter (TFF) - 1...,unknown,None
7,V113063002,99965692_fullbatch5,success,True,Wholesale Fibre BS2 (Chorus),WS Tail+Data - BS2 Res Fibre Starter (Chorus) ...,unknown,None
8,V113063051,99965692_fullbatch5,success,True,Wholesale Fibre BS2 (Chorus),WS Tail+Data - BS2 Res Fibre Starter (Chorus) ...,unknown,None
9,V113063010,99965692_fullbatch5,success,True,Wholesale Fibre BS2 (Chorus),WS Tail+Data - BS2 Res Fibre Starter (Chorus) ...,unknown,None


## 8. Save

In [16]:
save_df("order_results", order_results_df)


Saved 247 rows -> migration_data\07_order_creation_results.csv
